In [1]:
from src.backend import TorchBackend
from src.kvector import compute_k0xy, compute_Kxy
from src.geometry import Rectangle, Bitmap, Canvas, VectorObject
from src.layer import Layer
from src.source import Source
from src.sim import Eigensolver

from tests.backend import test_torch_backend_full
from tests.kvector import test_k0xy_full, test_Kxy_full
from tests.geometry import run_all_geometry_tests
import torch

import matplotlib.pyplot as plt
test_k0xy_full()
test_Kxy_full()
run_all_geometry_tests()

✓ full kvector test passed.
✓ full Kxy test passed.
FFT vs analytic RMSE: 0.0006081252346614848
All geometry tests passed.


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64
backend = TorchBackend(device=device, dtype=dtype)

wavelength = torch.nn.Parameter(torch.tensor([500.0, 600.0, 700.0, 100, 600]), requires_grad=True)  # in nm
theta_inc = torch.nn.Parameter(torch.tensor([30.0, 40.0, 50.0, 60.0]) * torch.pi / 180, requires_grad=True)  # in radians
phi_inc = torch.nn.Parameter(torch.tensor([45.0, 55.0, 65.0]) * torch.pi / 180, requires_grad=True)  # in radians
src = Source(backend, wavelength=wavelength, theta=theta_inc, phi=phi_inc)

print(src.wavelength)
print(src.theta)
print(src.phi)

n_inc = 2+1j
k0x, k0y = compute_k0xy(backend, wavelength, theta_inc, phi_inc, n_inc)
print("k0x:", k0x.shape)
print("k0y:", k0y.shape)

k0x_sum = k0x.sum()
k0y_sum = k0y.sum()
k0x_sum.backward(retain_graph=True)
k0y_sum.backward()

print("d k0x / d theta_inc:", theta_inc.grad)
print("d k0y / d phi_inc:", phi_inc.grad)
print("d k0x / d wavelength:", wavelength.grad)

tensor([500., 600., 700., 100., 600.], device='cuda:0', dtype=torch.float64,
       grad_fn=<ToCopyBackward0>)
tensor([0.5236, 0.6981, 0.8727, 1.0472], device='cuda:0', dtype=torch.float64,
       grad_fn=<ToCopyBackward0>)
tensor([0.7854, 0.9599, 1.1345], device='cuda:0', dtype=torch.float64,
       grad_fn=<ToCopyBackward0>)
k0x: torch.Size([5, 4, 3])
k0y: torch.Size([5, 4, 3])
d k0x / d theta_inc: tensor([0.7545, 0.6674, 0.5600, 0.4356])
d k0y / d phi_inc: tensor([ 0.0000, -0.1435, -0.2827])
d k0x / d wavelength: tensor([-0.0006, -0.0004, -0.0003, -0.0144, -0.0004])


/home/rodionsa/Code/MetaSolverRCWA/src/backend.py:507: UserWarning: Casting complex values to real discards the imaginary part (Triggered internally at /pytorch/aten/src/ATen/native/Copy.cpp:309.)
  return t.to(device=self.device, dtype=target_dtype)


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64
backend = TorchBackend(device=device, dtype=dtype)

wavelength = torch.nn.Parameter(torch.tensor([500.0, 600.0, 700.0, 100, 600]), requires_grad=True)  # in nm
theta_inc = torch.nn.Parameter(torch.tensor([30.0, 40.0, 50.0, 60.0]) * torch.pi / 180, requires_grad=True)  # in radians
phi_inc = torch.nn.Parameter(torch.tensor([45.0, 55.0, 65.0]) * torch.pi / 180, requires_grad=True)  # in radians
src = Source(backend, wavelength=wavelength, theta=theta_inc, phi=phi_inc)

print(src.wavelength)
print(src.theta)
print(src.phi)

n_inc = torch.nn.Parameter(torch.tensor([2.0, 3.0, 4.0, 5.0, 6.0]), requires_grad=True)
Kx, Ky = src.Kxy(n_inc, M=1, N=1, canvas=Canvas(period=(1.0, 1.0), grid=(10, 10)))

print("Kx:", Kx.shape)
print("Ky:", Ky.shape)

Kx_sum = Kx.sum()
Ky_sum = Ky.sum()
Kx_sum.backward(retain_graph=True)
Ky_sum.backward()   
print("d Kx / d theta_inc:", theta_inc.grad)
print("d Ky / d phi_inc:", phi_inc.grad)
print("d Kx / d wavelength:", wavelength.grad)
print("d Kx / d n_inc:", n_inc.grad)

tensor([500., 600., 700., 100., 600.], device='cuda:0', dtype=torch.float64,
       grad_fn=<ToCopyBackward0>)
tensor([0.5236, 0.6981, 0.8727, 1.0472], device='cuda:0', dtype=torch.float64,
       grad_fn=<ToCopyBackward0>)
tensor([0.7854, 0.9599, 1.1345], device='cuda:0', dtype=torch.float64,
       grad_fn=<ToCopyBackward0>)
Kx: torch.Size([5, 4, 3, 3, 3])
Ky: torch.Size([5, 4, 3, 3, 3])
d Kx / d theta_inc: tensor([644.7180, 570.2866, 478.5273, 372.2281])
d Ky / d phi_inc: tensor([   0.0000, -122.6587, -241.5905])
d Kx / d wavelength: None
d Kx / d n_inc: tensor([103.2880, 103.2880, 103.2880, 103.2880, 103.2880])


In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64
backend = TorchBackend(device=device, dtype=dtype)
backend2 = TorchBackend(device=device, dtype=dtype)

period = (1.0, 1.0)
grid = (10, 10)
canvas = Canvas(period=period, grid=grid)

wavelength = torch.nn.Parameter(torch.tensor([500.0, 600.0, 700.0, 100, 600]), requires_grad=True)  # in nm
theta_inc = torch.nn.Parameter(torch.tensor([30.0, 40.0, 50.0, 60.0]) * torch.pi / 180, requires_grad=True)  # in radians
phi_inc = torch.nn.Parameter(torch.tensor([45.0, 55.0, 65.0]) * torch.pi / 180, requires_grad=True)  # in radians
src = Source(backend, wavelength=wavelength, theta=theta_inc, phi=phi_inc)

center = torch.nn.Parameter(torch.tensor([0.5, 0.5]), requires_grad=True)
size = torch.nn.Parameter(torch.tensor([0.2, 0.2]), requires_grad=True)
epsilon = torch.nn.Parameter(torch.tensor(4.0), requires_grad=True)
mu = torch.nn.Parameter(torch.tensor([1.0, 1, 1, 1, 1]), requires_grad=True)

rect = Rectangle(backend, canvas, center=center, size=size, epsilon=epsilon, mu=mu)


epsilon_bg = torch.nn.Parameter(torch.tensor([1.0, 2, 1, 1, 1]), requires_grad=True)
mu_bg = torch.nn.Parameter(torch.tensor(1.0), requires_grad=True)
layer = Layer(rect, thickness=100, epsilon_bg=epsilon_bg, mu_bg=mu_bg)

solver = Eigensolver(layer, src)


TypeError: len() of a 0-d tensor

In [21]:
layer.epsilon

tensor(4.+0.j, device='cuda:0', dtype=torch.complex128,
       grad_fn=<ToCopyBackward0>)